In [1]:
%load_ext autoreload
%autoreload 2
from train import *
import matplotlib.pyplot as plt
from metrics_fns import *
import fourier_modules as FM

methods=['fomonms_size_3','FU','UFU','Vanilla','Attention']
metrics=['psnr','lpips','gatys','sifid','AC']
zooms=[1,2]

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/home/chatill243/anaconda3/envs/simulditex/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/chatill243/anaconda3/envs/simulditex/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /home/chatill243/anaconda3/envs/simulditex/lib/python3.12/site-packages/lpips/weights/v0.1/vgg.pth


/home/chatill243/anaconda3/envs/simulditex/lib/python3.12/site-packages/lpips/lpips.py:107: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch.load(mod

In [3]:
#100 synthesis
B=5
N=100



lists={(metric,method,zoom):[] for metric, method, zoom in itertools.product(metrics,methods,zooms)}

        
for i in tqdm(range(1,16)):
    for method in methods:
        for d in os.listdir('./runs'):
            if str(i) == d.split('_')[1] and method in d:
                if method=='FU' and 'UFU' in d:
                    break
                if 'fomo' in d and 'EMN_1_CA_1' in d:
                    name=d
                    break
                name=d
        
        for zoom in zooms:
            with torch.no_grad():
                parser = argparse.ArgumentParser()
                args ,_= parser.parse_known_args()
                args=load_args(os.path.join('./runs',name,'args.json'),args)
                FM.CA=args.CA
                FM.EMN=args.EMN
                dset=DS_rot(args.dataset_path,image_size=(args.img_size,args.img_size),min_mask_shape=(64,64),max_mask_shape=(200,200))
                loader=DataLoader(dset,batch_size=args.bs,shuffle=True,drop_last=True,num_workers=0)
                M = Model(nc_im=3,nc_start=args.dim,nc_max=512,depth=args.depth,fourier_mode=args.fourier_mode,train_size=(args.img_size,args.img_size),nms_size=args.nms_size).to(device)
                try:
                    M.load_state_dict(torch.load(os.path.join('runs',name,'M.pth')),strict=False) 
                except: 
                    print('Model not loaded!')
                    pass
                M.eval()          

                for j in range(N//B):   
                    torch.manual_seed(j)
                    loader = DataLoader(dset,batch_size=args.bs,shuffle=True,drop_last=True,num_workers=0)
                    x_plot,mask_plot=next(iter(loader))
                    x_plot,mask_plot=x_plot.to(device),mask_plot.to(device)
                    b,c,h,w=x_plot.shape

                    ratio=(zoom,zoom)
                    x_plot_zoom=F.pad(x_plot,(((ratio[1]-1)*w)//2,((ratio[1]-1)*w)-((ratio[1]-1)*w)//2,((ratio[0]-1)*h)//2,((ratio[0]-1)*h)-((ratio[0]-1)*h)//2))
                    mask_plot_zoom=F.pad(mask_plot,(((ratio[1]-1)*w)//2,((ratio[1]-1)*w)-((ratio[1]-1)*w)//2,((ratio[0]-1)*h)//2,((ratio[0]-1)*h)-((ratio[0]-1)*h)//2))
                    masked_plot_zoom=x_plot_zoom*mask_plot_zoom
                    n_plot_zoom=torch.randn_like(masked_plot_zoom)


                    rec=M(n_plot_zoom,mask_plot_zoom,masked_plot_zoom)   

                    if zoom==1:
                        lists['psnr',method,zoom].append(psnr(x_plot, rec))
                        lists['lpips',method,zoom].append(lpips_dist(x_plot, rec).mean())

                    input_torch = Variable(prep(x_plot)).cuda()
                    targets = [GramMatrix()(f).detach() for f in vgg(input_torch, loss_layers)]
                    im = Variable(prep(rec)).cuda()
                    out = vgg(im, loss_layers)
                    loss=sum([W[a]*loss_fns[a](f, targets[a]) for a,f in enumerate(out)])
                    lists['gatys',method,zoom].append(loss)

                    lists['sifid',method,zoom].append(sifid_2imgs(x_plot, rec))


                    c1=torch.fft.fft2(x_plot,norm='forward').abs()**2
                    c2=torch.fft.fft2(rec,norm='forward').abs()**2
                    c2=c2[...,::zoom,::zoom]
                    ac=(fft.ifft2((c1-c2),norm='forward').real**2).sum(dim=(1,2,3)).sqrt().mean()
                    lists['AC',method,zoom].append(ac)






100%|██████████| 15/15 [1:40:37<00:00, 402.53s/it]


In [6]:
for zoom in zooms:
    header, rows = build_table(lists, methods, metrics, zoom)
    print_ascii_table(header, rows, title=f"Zoom = {zoom}")

Zoom = 1
method         | psnr  | lpips | gatys     | sifid | AC   
---------------+-------+-------+-----------+-------+------
FOMONMS_SIZE_3 | 13.74 | 0.28  | 9375.05   | 0.18  | 5.43 
FU             | 12.97 | 0.37  | 23682.01  | 0.46  | 7.20 
UFU            | 12.64 | 0.33  | 13659.80  | 0.22  | 6.52 
VANILLA        | 12.16 | 0.53  | 216191.77 | 3.28  | 13.59
ATTENTION      | 12.55 | 0.40  | 58053.69  | 1.02  | 9.80 

Zoom = 2
method         | gatys     | sifid | AC   
---------------+-----------+-------+------
FOMONMS_SIZE_3 | 11107.32  | 0.29  | 6.13 
FU             | 57878.45  | 1.14  | 11.19
UFU            | 372833.59 | 6.35  | 17.24
VANILLA        | 265267.44 | 4.48  | 18.13
ATTENTION      | 374993.75 | 6.17  | 24.80



In [7]:
for zoom in zooms:
    header, rows = build_table(lists, methods, metrics, zoom)
    print_latex_table(
        header, rows,
        caption=f"Results for zoom={zoom}",
        label=f"tab:zoom{zoom}"
    )

\begin{table}[htb]
\centering
\resizebox{.8\linewidth}{!}{%
\begin{tabular}{lccccc}
\toprule
Method & PSNR $\uparrow$ & LPIPS $\downarrow$ & GATYS ($\times 10^3$) $\downarrow$ & SIFID $\downarrow$ & AC $\downarrow$ \\
\midrule
FOMONMSSIZE3 & \textbf{13.74} & \textbf{0.28} & \textbf{9.4} & \textbf{0.18} & \textbf{5.43} \\
FU & 12.97 & 0.37 & 23.7 & 0.46 & 7.20 \\
UFU & 12.64 & 0.33 & 13.7 & 0.22 & 6.52 \\
VANILLA & 12.16 & 0.53 & 216.2 & 3.28 & 13.59 \\
ATTENTION & 12.55 & 0.40 & 58.1 & 1.02 & 9.80 \\
\bottomrule
\end{tabular}%
}
\captionof{table}{Results for zoom=1}
\label{tab:zoom1}
\end{table}

\begin{table}[htb]
\centering
\resizebox{.8\linewidth}{!}{%
\begin{tabular}{lccc}
\toprule
Method & GATYS ($\times 10^3$) $\downarrow$ & SIFID $\downarrow$ & AC $\downarrow$ \\
\midrule
FOMONMSSIZE3 & \textbf{11.1} & \textbf{0.29} & \textbf{6.13} \\
FU & 57.9 & 1.14 & 11.19 \\
UFU & 372.8 & 6.35 & 17.24 \\
VANILLA & 265.3 & 4.48 & 18.13 \\
ATTENTION & 375.0 & 6.17 & 24.80 \\
\bottomrule
\end{t

# Ablation table

In [2]:
B = 5
N = 100

configs = [
    "CA=0_EMN=0",
    "CA=0_EMN=1",
    "CA=1_EMN=0",
    "CA=1_EMN=1",
]

metrics = ["gatys", "sifid", "AC"]

lists = {(cfg, m): [] for cfg in configs for m in metrics}


for name in tqdm(['img_'+str(i) for i in range(1,16)]):
    for CA in [0, 1]:
        for EMN in [0,1]:
            FM.EMN = EMN
            FM.CA = CA  

            cfg_name = f"CA={CA}_EMN={EMN}"
            setup =  f"_EMN_{EMN}_CA_{CA}"
            parser = argparse.ArgumentParser()
            args,_ = parser.parse_known_args()
            args = load_args(os.path.join('./runs', name+'_fomonms_size_3'+setup,'args.json'), args)
            #print(args.name)
            dset = DS_rot(args.dataset_path,
                          image_size=(args.img_size,args.img_size),
                          min_mask_shape=(64,64),
                          max_mask_shape=(200,200))
            loader = DataLoader(dset,batch_size=args.bs,shuffle=True,drop_last=True,num_workers=0)

            M = Model(nc_im=3,nc_start=args.dim,nc_max=512,depth=args.depth,
                      fourier_mode=args.fourier_mode,
                      train_size=(args.img_size,args.img_size),
                      nms_size=args.nms_size).to(device)

            try:
                M.load_state_dict(torch.load(os.path.join('runs',name+'_fomonms_size_3'+setup,'M.pth')), strict=False)
            except:
                print('model not loaded!')
                continue

            M.eval()

            for j in range(N//B):
                torch.manual_seed(j)
                loader = DataLoader(dset,batch_size=args.bs,shuffle=True,drop_last=True,num_workers=0)
                x_plot, mask_plot = next(iter(loader))
                x_plot, mask_plot = x_plot.to(device), mask_plot.to(device)

                b,c,h,w = x_plot.shape

                ratio = (2,2)
                x_plot2x = F.pad(x_plot,(((ratio[1]-1)*w)//2,((ratio[1]-1)*w)-((ratio[1]-1)*w)//2,
                                          ((ratio[0]-1)*h)//2,((ratio[0]-1)*h)-((ratio[0]-1)*h)//2))
                mask_plot2x = F.pad(mask_plot,(((ratio[1]-1)*w)//2,((ratio[1]-1)*w)-((ratio[1]-1)*w)//2,
                                                ((ratio[0]-1)*h)//2,((ratio[0]-1)*h)-((ratio[0]-1)*h)//2))

                masked_plot2x = x_plot2x * mask_plot2x
                n_plot2x = torch.randn_like(masked_plot2x)

                with torch.no_grad():
                    rec = M(n_plot2x, mask_plot2x, masked_plot2x)

                # ---- metrics (same as your code) ----
                #lists[(cfg_name,"psnr")].append(psnr(x_plot, rec))
                #lists[(cfg_name,"lpips")].append(lpips_dist(x_plot, rec).mean())

                input_torch = Variable(prep(x_plot)).cuda()
                targets = [GramMatrix()(f).detach() for f in vgg(input_torch, loss_layers)]
                im = Variable(prep(rec)).cuda()
                out = vgg(im, loss_layers)
                loss = sum([W[a]*loss_fns[a](f, targets[a]) for a,f in enumerate(out)])
                lists[(cfg_name,"gatys")].append(loss)

                lists[(cfg_name,"sifid")].append(sifid_2imgs(x_plot, rec))

                c1 = torch.fft.fft2(x_plot,norm='forward').abs()**2
                c2 = torch.fft.fft2(rec,norm='forward').abs()**2
                c2 = c2[...,::2,::2]
                ac = (fft.ifft2((c1-c2),norm='forward').real**2).sum(dim=(1,2,3)).sqrt().mean()
                lists[(cfg_name,"AC")].append(ac)

100%|██████████| 15/15 [44:00<00:00, 176.01s/it]


In [3]:
header, rows = build_ablation_table(lists, configs, metrics)
print_ascii_table(header, rows, title="Ablation")

Ablation
method     | gatys    | sifid | AC   
-----------+----------+-------+------
CA=0_EMN=0 | 41398.33 | 0.52  | 14.68
CA=0_EMN=1 | 14195.88 | 0.31  | 6.48 
CA=1_EMN=0 | 43098.20 | 0.62  | 11.16
CA=1_EMN=1 | 11093.48 | 0.29  | 6.13 



In [4]:
header, rows = build_ablation_table(lists, configs, metrics)
print_latex_ablation_table(
    header,
    rows,
    caption="Ablation study on CA and HPF components.",
    label="tab:ablation"
)

\begin{table}[htb]
\centering
\resizebox{.8\linewidth}{!}{%
\begin{tabular}{lccc}
\toprule
Method & GATYS ($\times 10^3$) $\downarrow$ & SIFID $\downarrow$ & AC $\downarrow$ \\
\midrule
CA=0, HPF=0 & 41.4 & 0.52 & 14.68 \\
CA=0, HPF=1 & 14.2 & 0.31 & 6.48 \\
CA=1, HPF=0 & 43.1 & 0.62 & 11.16 \\
CA=1, HPF=1 & \textbf{11.1} & \textbf{0.29} & \textbf{6.13} \\
\bottomrule
\end{tabular}%
}
\captionof{table}{Ablation study on CA and HPF components.}
\label{tab:ablation}
\end{table}

